# Spambase

First experiment using the shared workflow in `ml_workflow.py`.
The goal is to classify emails as spam (`1`) or non-spam (`0`).


In [1]:
import pandas as pd

from ml_workflow import (
    evaluate_classifier,
    load_spambase,
    split_dataset,
    train_logistic_baseline,
)


## Dataset

We check the number of samples, the number of features, missing values,
and the class distribution.


In [2]:
X, y = load_spambase()

print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Missing feature values: {X.isna().sum().sum()}")


Samples: 4601
Features: 57
Missing feature values: 0


In [3]:
class_distribution = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "proportion": y.value_counts(normalize=True).sort_index(),
})
class_distribution.index.name = "spam"
class_distribution.round(3)


,count,proportion
spam,,
0,2788,0.606
1,1813,0.394


## Train/test split

We use 80% of the data for training and 20% for testing.
The split is stratified to preserve similar class proportions.
The random seed is fixed at 42.


In [4]:
X_train, X_test, y_train, y_test = split_dataset(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")


Training samples: 3680
Test samples: 921


## Logistic regression

The pipeline applies median imputation and standardization before
logistic regression. The preprocessing steps are fitted on the training split.


In [5]:
model = train_logistic_baseline(X_train, y_train)


## Test results

Precision, recall, and F1-score refer to the spam class (`1`).


In [6]:
scores = evaluate_classifier(model, X_test, y_test)
results = pd.Series(scores, name="Logistic regression").to_frame()
results.round(3)


,Logistic regression
accuracy,0.929
balanced_accuracy,0.924
precision,0.921
recall,0.898
f1,0.909
roc_auc,0.970
average_precision,0.951


On this split, logistic regression reaches about 92.9% accuracy and an F1-score of 0.909. Precision is 92.1% and recall is 89.8% for spam.

This gives a first reference for later model comparisons. Model and feature selection will use cross-validation on the training split.


## Cross-validation

We evaluate the pipeline using five stratified folds of the training split.
The table reports the mean and standard deviation of each metric.

In [7]:
from ml_workflow import cross_validate_classifier

cv_scores = cross_validate_classifier(model, X_train, y_train)
cv_scores.agg(["mean", "std"]).T.round(3)

,mean,std
accuracy,0.923,0.012
balanced_accuracy,0.915,0.012
precision,0.920,0.023
recall,0.881,0.019
f1,0.900,0.015
roc_auc,0.971,0.007
average_precision,0.947,0.015


## Class weighting

In [8]:
balanced_model = train_logistic_baseline(
    X_train, y_train, class_weight="balanced"
)

balanced_cv_scores = cross_validate_classifier(
    balanced_model, X_train, y_train, random_state=42
)

comparison = pd.DataFrame({
    "standard": cv_scores.mean(),
    "balanced": balanced_cv_scores.mean(),
}).T

comparison.round(3)

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
standard,0.923,0.915,0.920,0.881,0.900,0.971,0.947
balanced,0.928,0.925,0.903,0.914,0.909,0.971,0.947


Class weighting increases recall from 0.881 to 0.914, while precision decreases from 0.920 to 0.903. F1 improves slightly from 0.900 to 0.909. The weighted model detects more spam at the cost of additional false positives.

## Comparing classifiers

In [ ]:
from ml_workflow import build_models, compare_models

SEED = 42  

models = build_models(random_state=SEED)

comparison, fold_scores = compare_models(
    models,
    X_train,
    y_train,
    random_state=SEED,
)

mean_scores = comparison.xs("mean", axis=1, level=1)
mean_scores.sort_values("f1", ascending=False).round(3)